# Unit 8 exercises: measure the car

A model's numbers can be measured instead of guessed.
These exercises fit the longitudinal map with least squares, choose its lag by the residual and check it on rows it did not use, solve the same problem with QR, measure the steering gain, measure acceleration from positions, run grip probes of your own, and put the measurements to work.
Every number here is a number about racekit's **carla-like car**: a model, not the truth, and not CARLA.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise are reused in later ones, so run the exercises in order.
Every race uses `carla_like(noise=True)` (unless an exercise says otherwise) with the stated seed; a race takes a few seconds, and the exercises keep the number of races small.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import carla_like
from racekit.runner import run_race
from racekit.sysid_data import load_log
from racekit.tracks import grip_sections, load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_07, UNIT_08 = AGENTS / "unit-07.py", AGENTS / "unit-08.py"
workdir = Path(mkdtemp(prefix="roar-unit-08-ex-"))
DT = 0.05  # s: one control tick, one log or telemetry row
WHEELBASE = 2.875  # m, L (Unit 3)
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on the max |cross-track error|
ASSUMED = (6.0, 8.0, 4.0e-4)  # agents/unit-07.py's A_THROTTLE, A_BRAKE, C_DRAG
MU_G_TEST_CIRCUIT = (7.89, 9.72, 8.65, 7.87, 8.30, 8.14)  # agents/unit-08.py's MU_G (m/s^2)


def variant(source, name, **constants):
    # Unit 1's idiom: a copy of a submission with some module constants changed, written into workdir.
    return write_variant(source, workdir / f"{name}.py", {key: repr(value) for key, value in constants.items()})


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The clean rule: finished, 0 collisions, and max |cross-track error| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET


race_log, race_meta = load_log("monza-like-seed0.csv")  # a full noisy race on monza-like
steady_log, steady_meta = load_log("oval-steady-seed0.csv")  # one lap of the oval at a steady 15 m/s

## Exercise 1 - Least squares on six rows

A toy log of six steps of the longitudinal model dv/dt = a_T·throttle − a_B·brake − c·v|v| (no lag):

| step | throttle | brake | mean speed v̄ (m/s) | measured dv/dt (m/s²) |
|---|---|---|---|---|
| 0 | 1.0 | 0.0 | 10 | 5.02 |
| 1 | 0.6 | 0.0 | 20 | 2.85 |
| 2 | 0.0 | 0.5 | 30 | −4.75 |
| 3 | 0.0 | 1.0 | 25 | −9.2275 |
| 4 | 0.3 | 0.0 | 15 | 1.4425 |
| 5 | 1.0 | 0.0 | 5 | 4.9725 |

Row k of the matrix is (throttle_k, −brake_k, −v̄_k·|v̄_k|) and the unknowns are (a_T, a_B, c).

Define:

- `A1`: the 6 × 3 matrix (a NumPy array), and `b1`: the six measured rates;
- `c1`: the least-squares solution, `np.linalg.lstsq(A1, b1, rcond=None)[0]`;
- `residuals1`: A1·c1 − b1 (six numbers), and `rms1`: the root of their mean square (m/s²);
- `sse1`: the sum of squared residuals at `c1`;
- `perpendicular`: the largest |entry| of A1ᵀ·`residuals1` (the residual is perpendicular to every column of A1 at the least-squares solution, so this should be tiny);
- `nudged_sse`: a list of six sums of squared residuals, one for each way of changing **one** coefficient of `c1` by +1% or −1% (a_T up, a_T down, a_B up, a_B down, c up, c down, in that order);
- `always_worse`: `True` when every entry of `nudged_sse` is larger than `sse1`;
- `why_c_is_hard`: one or two sentences: the drag c is the least certain of the three numbers here (compare `c1[2]` with the value the six rows were built from, 3.0e-4, if you like); which column of A1 is to blame, and what would a better log need?

In [ ]:
A1 = None  # TODO
b1 = None  # TODO
c1 = None  # TODO
residuals1 = None  # TODO
rms1 = None  # TODO
sse1 = None  # TODO
perpendicular = None  # TODO
nudged_sse = []  # TODO
always_worse = None  # TODO
why_c_is_hard = "TODO"

## Exercise 2 - The lag, and a fit checked on rows it never saw

Use the race log, `race_log` (columns `t, x, y, yaw, vx, vy, throttle, steer, brake`); the speed is `np.hypot(vx, vy)`.

Write `longitudinal_rows(log, lag)`: for every k from `lag` up to `len(log) - 2` such that rows k and k + 1 are one tick apart (|t[k+1] − t[k] − 0.05| < 1e-6) **and** rows k − lag and k are `lag` ticks apart (|t[k] − t[k−lag] − lag·0.05| < 1e-6), one equation: the row (throttle[k − lag], −brake[k − lag], −v̄·|v̄|) with v̄ the mean of the speeds at k and k + 1, and the rate (v[k+1] − v[k]) / 0.05.
It returns `(A, b, k)`, with `k` the array of the rows used.

Define:

- `longitudinal_rows`: the function;
- `lag_rms`: a dict from each lag 0, 1, 2, 3, 4, 5 to the residual RMS of its least-squares fit (m/s²);
- `best_lag`: the lag with the smallest RMS;
- `second_half_fit`: the coefficients (a_T, a_B, c) fitted at `best_lag` on the rows with `t[k] >= t[-1] / 2` only (the second half of the race);
- `first_half_rms_fit`: the RMS of `second_half_fit` on the **other** rows (the first half);
- `first_half_rms_assumed`: the RMS of the assumed map `ASSUMED` on the same first-half rows;
- `full_fit`: the coefficients fitted at `best_lag` on all the rows;
- `percent_apart`: a list of the three differences 100·(second_half_fit − full_fit) / full_fit (%);
- `fig`, `ax`: one plot of `lag_rms` against the lag (markers and a line), with axis labels and a title;
- `what_the_lag_is`: one sentence: what does the best lag say about the car, in seconds?

In [ ]:
def longitudinal_rows(log, lag):
    return None, None, None  # TODO: (A, b, k)


lag_rms = {}  # TODO
best_lag = None  # TODO
second_half_fit = None  # TODO
first_half_rms_fit = None  # TODO
first_half_rms_assumed = None  # TODO
full_fit = None  # TODO
percent_apart = []  # TODO
fig, ax = None, None  # TODO
what_the_lag_is = "TODO"

## Exercise 3 - QR, and a log that cannot see the brake

Write `back_substitute(r, y)`: solve r·c = y for an upper-triangular m × m matrix r by back substitution (the last unknown first: c[m−1] = y[m−1] / r[m−1, m−1], then upwards, each row using the unknowns already found), returning a NumPy array.
Check it on r = [[2, 1, −1], [0, 3, 2], [0, 0, 4]] and y = [3, 13, 8] before you trust it.

Then, with Exercise 2's `longitudinal_rows(race_log, 1)` as (A, b):

- `q`, `r` = `np.linalg.qr(A)`;
- `c_qr` = `back_substitute(r, q.T @ b)`;
- `largest_gap`: the largest |`c_qr` − the `lstsq` solution|, relative to the `lstsq` solution;
- `constant` = ‖b‖² − ‖qᵀb‖²;
- `identity_errors`: for each of the five coefficient vectors `c_qr`, `ASSUMED`, (0, 0, 0), (5, 9, 0) and (1, 1, 1), the |difference| between ‖Ac − b‖² and ‖rc − qᵀb‖² + `constant`, divided by ‖Ac − b‖² (a list of five numbers).

Now do the same fit on the **steady** log, `steady_log`, with the same function (lag 1).
That lap holds a steady 15 m/s: the speed loop barely brakes.

- `steady_fit`: its least-squares (a_T, a_B, c);
- `race_r_diagonal`, `steady_r_diagonal`: the three diagonal entries of r for the race log and for the steady log (`[r[i, i] for i in range(3)]`);
- `excitation`: two or three sentences: compare the brake entry of the two diagonals and the two fitted a_B; what does a small diagonal entry of r mean for back substitution, and what does a log need to contain before a coefficient can be identified from it?

In [ ]:
def back_substitute(r, y):
    return None  # TODO: a NumPy array


small_check = None  # TODO: back_substitute on the 3 x 3 example
q, r = None, None  # TODO
c_qr = None  # TODO
largest_gap = None  # TODO
constant = None  # TODO
identity_errors = []  # TODO
steady_fit = None  # TODO
race_r_diagonal, steady_r_diagonal = None, None  # TODO
excitation = "TODO"

## Exercise 4 - The steering gain, and which rows to trust

Write `steering_gain(log, lag=1, stillness_limit=0.002)` for a sensor log:

- the yaw rate over k → k + 1 is the wrapped heading change, ((yaw[k+1] − yaw[k] + π) mod 2π − π) / 0.05;
- the speed is the mean of `np.hypot(vx, vy)` at k and k + 1;
- the steer paired with k → k + 1 is the one logged at k − lag (0 for the first `lag` transitions);
- `stillness` at k is the standard deviation of that paired steer over k − 10 … k (fewer at the start);
- a row counts when the speed is above 5 m/s, |steer| > 0.005 and `stillness < stillness_limit`;
- the wheel angle of a counted row is atan(r·L / v), with L = `WHEELBASE` (`math.atan2(r * WHEELBASE, v)`), and the gain is the one-column least-squares fit of wheel angle = gain · (−steer).

It returns `(the number of counted rows, the gain in degrees)`, and `(count, None)` when 20 rows or fewer count.

Define:

- `steering_gain`: the function;
- `steady_gain`: a dict from each lag 0, 1, 2 to the gain on `steady_log` (degrees, `stillness_limit` 0.002);
- `race_gain`: a dict from each `stillness_limit` 0.002, 0.01 and 0.05 to the result `(rows, gain)` on `race_log` at lag 1;
- `trusted`: the steady log's gain at lag 1;
- `why_still`: two or three sentences: why does the race log's gain fall as the limit is loosened, and why is the steady log the one to trust? (Think about what the tyres do just after the steer changes, Unit 6.)

Then the size of the error the assumed 70° makes.
The controller wants a wheel angle of 5°, so it sends steer = −5 / 70; the car turns its wheels −steer × `trusted` degrees.

- `wheel_angle_got`: that wheel angle (degrees);
- `radius_wanted`, `radius_got`: the turning radii L / tan(δ) of the two wheel angles (m).

In [ ]:
def steering_gain(log, lag=1, stillness_limit=0.002):
    return 0, None  # TODO: (rows, gain in degrees or None)


steady_gain = {}  # TODO
race_gain = {}  # TODO
trusted = None  # TODO
why_still = "TODO"
wheel_angle_got = None  # TODO
radius_wanted, radius_got = None, None  # TODO

## Exercise 5 - Acceleration from positions

**First, check the formula where you know the answer.**
A car drives a circle of radius R = 100 m at v = 20 m/s, and its position is sampled every 0.05 s for 10 s (201 samples): at time t_j = 0.05·j the position is (R cos(v t_j / R), R sin(v t_j / R)).

- `circle_accel`: the |a| of the second difference, (p_{j+1} − 2p_j + p_{j−1}) / 0.05², at every inner sample (a NumPy array of 199 values);
- `circle_exact`: v² / R;
- `circle_error`: the largest |`circle_accel` − `circle_exact`| / `circle_exact`.

**Then write the probe's row rule** as a function, for racekit telemetry (`result.telemetry`):
`section_p99(telemetry, track, min_rows=20)` returns a list of six pairs `(rows, p99)`, one per grip section, where:

- |a| at row i (for i from 1 to N − 2) is the second difference of (x, y) at rows i − 1, i, i + 1;
- row i qualifies when rows i − 1, i and i + 1 all have `held == 0`, neither of the two steps between them is longer than 5 m, and `v` at row i is above 5 m/s;
- row i's section is `grip_sections(track)[(progress[i] + 1) % n]` (Unit 7: racekit's progress p is waypoint p + 1);
- `rows` is the number of qualifying rows in the section, and `p99` the 99th percentile of their |a| (`np.percentile(values, 99)`), or `None` when `rows <= min_rows`.

Race `agents/unit-08.py` on `"test-circuit"`, `seed=0`, `max_seconds=450.0` (a clean race), and define:

- `section_p99`: the function;
- `clean_race`: the `RaceResult`;
- `clean_race_p99`: `section_p99(clean_race.telemetry, load_track("test-circuit"))`;
- `fraction_of_measured`: a list of the six ratios p99 / `MU_G_TEST_CIRCUIT[s]`;
- `any_above`: `True` when any ratio is above 1;
- `lower_bound`: two sentences: what does `clean_race_p99` tell you about the grip, and why can it not be an estimate of it, however many clean races you log?

In [ ]:
circle_accel = None  # TODO
circle_exact = None  # TODO
circle_error = None  # TODO


def section_p99(telemetry, track, min_rows=20):
    return []  # TODO: six (rows, p99 or None) pairs


clean_race = None  # TODO
clean_race_p99 = None  # TODO
fraction_of_measured = []  # TODO
any_above = None  # TODO
lower_bound = "TODO"

## Exercise 6 - Two probes of your own

Probe two sections of `test-circuit-seed1`, a track of the training family: sections **2** and **5**.
A probe is a variant of `agents/unit-07.py` with `A_LAT = 3.5` and `GRIP` = 1.0 everywhere except 3.5 in the pushed section, raced on `"test-circuit-seed1"` with `laps=1`, `max_seconds=300.0`, `seed=0`.
Use Exercise 5's `section_p99` and keep only the pushed section's pair.

The lesson's rules: a probe that crashed gives an **estimate**; a probe that never crashed gives a **lower bound**; 20 qualifying rows or fewer give **no estimate**.

Define:

- `probes`: a dict from the section (2 and 5) to its `RaceResult`;
- `my_mu_g`: a dict from the section to its p99 (m/s², rounded to 0.01);
- `kind`: a dict from the section to `"estimate"`, `"lower bound"` or `"no estimate"`;
- `matches_record`: `True` when both values equal the lesson's table for `test-circuit-seed1` (9.36 and 7.56) to 0.01.

Now what the numbers buy.
With κ the Menger curvature of `test-circuit-seed1`'s waypoints (cyclic, as in Unit 2), find the **tightest** waypoint of each of the two sections (the largest |κ| among the waypoints whose `grip_sections` value is that section), and compare the corner cap √(a / |κ|) for three lateral limits a:

- `tightest`: a dict from the section to (that waypoint's index, its |κ| in 1/m);
- `cap_unit_7`: a dict from the section to the cap with Unit 7's a = 4.6 (m/s);
- `cap_unit_8`: a dict from the section to the cap with a = 0.56 · `my_mu_g[section]` (m/s);
- `cap_full`: a dict from the section to the cap with a = `my_mu_g[section]`, all of the measured grip (m/s);
- `which_section_gains`: one or two sentences: in which section does Unit 8's cap rise above Unit 7's, in which does it fall, and why is a fall here not a loss?

In [ ]:
probes = {}  # TODO
my_mu_g = {}  # TODO
kind = {}  # TODO
matches_record = None  # TODO
tightest = {}  # TODO
cap_unit_7 = {}  # TODO
cap_unit_8 = {}  # TODO
cap_full = {}  # TODO
which_section_gains = "TODO"

## Exercise 7 - Which fix fixed it?

On `test-circuit-seed1`, race seed **1** is one where Unit 7's agent is not clean.
Two things changed between Unit 7 and Unit 8: the steering gain (70° → 66°) and the grip (A_LAT → `MU_FRACTION` · `MU_G`).
Separate them with three races on `"test-circuit-seed1"`, `seed=1`, `max_seconds=450.0`:

- `unit_7`: `agents/unit-07.py` as shipped;
- `unit_7_at_66`: `agents/unit-07.py` with `MAX_STEER = math.radians(66.0)` only;
- `unit_8`: `agents/unit-08.py` with `test-circuit-seed1`'s measured grip, `MU_G = (7.72, 9.80, 9.36, 8.38, 9.53, 7.56)` (the lesson's table), and its shipped f = 0.56.

Define:

- the three `RaceResult` objects, `unit_7`, `unit_7_at_66` and `unit_8`;
- `outcome`: a dict from `"unit-07"`, `"unit-07 at 66"` and `"unit-08"` to a tuple (clean, race time in s, max |cross-track error| in m);
- `steering_alone_enough`: `True` when `unit_7_at_66` is clean;
- `grip_gain_s`: the race time of `unit_7_at_66` minus that of `unit_8` (s), or `None` unless both are clean;
- `attribution`: two or three sentences: on this track and seed, which change made the agent clean and which made it faster? Would you expect the same answer on `test-circuit-seed2026` (the lesson's §5)?

In [ ]:
unit_7, unit_7_at_66, unit_8 = None, None, None  # TODO
outcome = {}  # TODO
steering_alone_enough = None  # TODO
grip_gain_s = None  # TODO
attribution = "TODO"

## Exercise 8 - Challenge: the drag from a coast-down alone

A classic way to measure drag: get the car moving, then **coast**, no throttle and no brake, and watch it slow down.
With throttle = brake = 0 the model is dv/dt = −c·v|v|: one unknown, one column.

The text below, appended to `agents/unit-08.py`, makes a submission that switches the speed loop off from 30 s to 50 s of the race: it still steers, but it commands throttle 0 and brake 0.
(It wraps the agent's own speed loop, `self.speed_pid.update`, so that it returns 0 in the window.)

```python
COAST = '''

COAST_FROM, COAST_TO = 30.0, 50.0  # s: the speed loop is switched off (no throttle, no brake) in between
_Unit08 = RoarCompetitionSolution


class RoarCompetitionSolution(_Unit08):
    async def initialize(self) -> None:
        await super().initialize()
        self.steps = 0
        speed_loop = self.speed_pid.update

        def coasting(error, dt, **kwargs):
            command = speed_loop(error, dt, **kwargs)
            return 0.0 if COAST_FROM <= self.steps * 0.05 < COAST_TO else command

        self.speed_pid.update = coasting

    async def step(self) -> None:
        await super().step()
        self.steps += 1
'''
```

Write that submission to `workdir / "coast.py"` (the text of `agents/unit-08.py` followed by `COAST`, as Unit 1 built its saboteur), and race it on `"test-circuit"`, `laps=1`, `max_seconds=120.0`, `seed=0`, twice: once with `car=carla_like(noise=False)` and once with `car=carla_like(noise=True)`.

For each race, the coast equations are the pairs k → k + 1 where rows k − 1, k and k + 1 are driving rows (`held == 0`) whose logged throttle and brake are both exactly 0 at k − 1 (the lag of Exercise 2) and k; each gives the row (−v̄·|v̄|) and the rate (v[k+1] − v[k]) / 0.05, with v the telemetry's `v`.

Define:

- `coast_races`: a dict from `"quiet"` (noise=False) and `"noisy"` (noise=True) to the `RaceResult`;
- `coast_rows`: a dict from the same keys to the number of coast equations;
- `coast_drag`: a dict from the same keys to the fitted c (1/m);
- `full_fit_drag`: the c of Exercise 2's `full_fit`;
- `compare`: three or four sentences: compare the quiet coast-down's c with the full fit's, and the noisy one's with both. Why can a coast-down on the noisy car be so far off? (The noise is a jitter added to the throttle the car applies, and a throttle cannot go below 0.) What does the full fit have that the coast-down lacks, and what else, besides drag, slows a car that coasts through corners?

In [ ]:
coast_races = {}  # TODO
coast_rows = {}  # TODO
coast_drag = {}  # TODO
full_fit_drag = None  # TODO
compare = "TODO"

## Exercise 9 - Challenge: a gentler probe

The lesson's probe pushes a section with `GRIP[s] = 3.5`, a planned 12.25 m/s² of cornering.
Would a gentler push do?
Probe sections **0** and **1** of `"test-circuit"` with `GRIP[s] = 2.0` instead (a planned 7.0 m/s²), everything else as in the lesson (`A_LAT = 3.5`, `laps=1`, `max_seconds=300.0`, `seed=0`), and analyse them with Exercise 5's `section_p99`.

Define:

- `gentle`: a dict from the section (0 and 1) to its `RaceResult`;
- `gentle_mu_g`: a dict from the section to the pushed section's p99 (m/s², rounded to 0.01);
- `gentle_kind`: a dict from the section to `"estimate"`, `"lower bound"` or `"no estimate"`, by the lesson's rules;
- `shift`: a dict from the section to `gentle_mu_g[s] - MU_G_TEST_CIRCUIT[s]` (m/s²; the shipped values come from the 3.5 probes);
- `gentle_verdict`: three or four sentences: did both sections still slide? One of the two numbers is close to the 3.5 probe's but the rules call it something else: is that rule too strict, or is it doing its job? When would a gentler probe be the better choice, on a real car?

In [ ]:
gentle = {}  # TODO
gentle_mu_g = {}  # TODO
gentle_kind = {}  # TODO
shift = {}  # TODO
gentle_verdict = "TODO"